In [ ]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import arcpy
import os
import csv
from arcpy import metadata as md

### Test if 'metadata' from 'arcpy' can extract metadata

In [4]:
print(dir(meta))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__from_scripting_arc_object__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_arc_object', 'accessConstraints', 'copy', 'credits', 'deleteContent', 'description', 'exportMetadata', 'importMetadata', 'isReadOnly', 'maxScale', 'minScale', 'reload', 'save', 'saveAsUsingCustomXSLT', 'saveAsXML', 'summary', 'synchronize', 'tags', 'thumbnailUri', 'title', 'upgrade', 'uri', 'xMax', 'xMin', 'xml', 'yMax', 'yMin']


In [5]:
gdb = r"Y:/aa_GSS_Student_Engagement_Training/Sho/GeoDBToText/data/Peatland_Watershed_Protection_Areas_(WPAs)"

arcpy.env.workspace = gdb

fc = arcpy.ListFeatureClasses()[0]

meta = md.Metadata(fc)

print(f"Title:\n{meta.title}\n")
print(f"Tags:\n{meta.tags}\n")
print(f"Summary:\n{meta.summary}\n")
print(f"Description:\n{meta.description}\n")
print(f"Credits:\n{meta.credits}\n")
print(f"Use Limitations:\n{meta.accessConstraints}")

Title:
Peatland Watershed Protection Areas (WPAs)

Tags:
peatland, SNA, water, Minnesota, MN

Summary:
WPAs serve as protective buffers around peatland core areas to minimize hydrologic and ecological impacts from adjacent land uses. This layer supports planning, permitting, ecological assessment, and peatland protection work across the Department of Natural Resources. It should not be used for regulatory purposes.

Statutory Context
• The Minnesota Peatland Protection Act (Minn. Stat. 84.035-84.036) lists Peatland SNAs by county but does not include legal descriptions or reference WPAs.
• WPAs are recognized in Minn. R. 6132.2000, Subp. 4(C), which restricts mining within WPAs and incorporates the 1984 peatland report by reference as the authoritative source for WPAs. This report also explicitly states that peat mining is prohibited within WPAs.
• The Minnesota State Law Library holds a DNR volume titled "Legal description and maps of peatland protection management areas" (Dec. 3, 198

### Fix the original code to utilize arcpy.metadata

In [ ]:
# ---------------------------------------------------------------------
# Input parameters
# ---------------------------------------------------------------------


GeoDB = arcpy.GetParameterAsText(0)

# Optional fallback values if metadata is missing
DefaultTitle = arcpy.GetParameterAsText(1)
DefaultTags = arcpy.GetParameterAsText(2)
DefaultSummary = arcpy.GetParameterAsText(3)
DefaultDescription = arcpy.GetParameterAsText(4)
DefaultCredits = arcpy.GetParameterAsText(5)
DefaultUseLimitations = arcpy.GetParameterAsText(6)
DefaultContact = arcpy.GetParameterAsText(7)
Delimiter = arcpy.GetParameterAsText(8)

# Set workspace
arcpy.env.workspace = GeoDB

In [ ]:
# ---------------------------------------------------------------------
# Export Feature Class to JSON
# ---------------------------------------------------------------------


def ExportJSON(FeatureClass):

    try:

        JsonFile = os.path.join(os.path.dirname(arcpy.env.workspace),
                                FeatureClass + ".json")

        if os.path.exists(JsonFile):
            arcpy.AddMessage("File exists: " + JsonFile + ". Deleted")
            os.remove(JsonFile)

        arcpy.AddMessage("Exporting " + JsonFile)

        arcpy.FeaturesToJSON_conversion(
            FeatureClass,
            JsonFile,
            "NOT_FORMATTED"
        )

    except Exception as e:
        arcpy.AddMessage(
            "Export failed for FeatureClass: {} {}".format(
                FeatureClass, str(e)
            )
        )

In [64]:
# ---------------------------------------------------------------------
# Export Feature Class to CSV
# ---------------------------------------------------------------------

def ExportCSV(FeatureClass):

    try:

        # Read metadata from the feature class
        meta = md.Metadata(FeatureClass)

        DatasetTitle = meta.title or DefaultTitle
        Tags = meta.tags or DefaultTags
        Summary = meta.summary or DefaultSummary
        Description = meta.description or DefaultDescription
        Credits = meta.credits or DefaultCredits
        UseLimitations = meta.accessConstraints or DefaultUseLimitations
        DatasetContact = DefaultContact

        CSVPath = os.path.join(
            os.path.dirname(arcpy.env.workspace),
            FeatureClass + ".csv"
        )

        if os.path.exists(CSVPath):
            arcpy.AddMessage("File exists: " + CSVPath + ". Deleted")
            os.remove(CSVPath)

        CSVFile = open(CSVPath, "a", encoding="utf-8")

        fields = arcpy.ListFields(FeatureClass)

        field_names = [field.name for field in fields]
        cursor_fields = ["Shape@WKT"] + field_names

        CSVFile.write(f"Title:\n{DatasetTitle}\n\n")
        CSVFile.write(f"Tags:\n{Tags}\n\n")
        CSVFile.write(f"Summary:\n{Summary}\n\n")
        CSVFile.write(f"Description:\n{Description}\n\n")
        CSVFile.write(f"Credits:\n{Credits}\n\n")
        CSVFile.write(f"Use Limitations:\n{UseLimitations}\n\n")
        CSVFile.write(f"Dataset Contact:\n{DatasetContact}\n\n")
        
        spatial_ref = arcpy.Describe(FeatureClass).spatialReference
        CSVFile.write("Spatial Reference:\n"
                      f"Name: {spatial_ref.name}\n"
                      f"WKID: {spatial_ref.factoryCode}\n\n"
                      "Well-Known Text (WKT):\n"
                      f"{spatial_ref.exportToString()}\n\n"
                      )

        CSVFile.write("Purpose:\n"
                      "This file is a human and machine readable equivalent of the layer\n"
                      f">>>{FeatureClass}\n"
                      "exported from the ESRI personal geodatabase\n"
                      f">>>{os.path.basename(GeoDB)}\n"
                      "and was generated to back up and archive the parent dataset\n"
                      "for posterity in a non-proprietary text format.\n\n"
                      )

        CSVFile.write("Note:\n"
                      "Row field values are separated by a pipe character |"
                      "to avoid confusion with commas in WKT geometry.\n\n"
                      )

        CSVFile.write(Delimiter.join(cursor_fields) + "\n")

        with arcpy.da.SearchCursor(FeatureClass, cursor_fields) as cursor:
            for row in cursor:
                CSVFile.write(
                    Delimiter.join("" if v is None else str(v) for v in row)
                )
                CSVFile.write("\n")

        CSVFile.close()

    except Exception as e:
        arcpy.AddMessage(
            "Export failed for FeatureClass: {} {}".format(
                FeatureClass, str(e)
            )
        )

In [65]:
# ---------------------------------------------------------------------
# Process all feature classes
# ---------------------------------------------------------------------

try:

    FeatureClasses = arcpy.ListFeatureClasses()

    for FeatureClass in FeatureClasses:

        arcpy.AddMessage("Processing " + FeatureClass)

        ExportCSV(FeatureClass)
        ExportJSON(FeatureClass)

except Exception as e:

    arcpy.AddMessage("Error: " + str(e))

### Test the above code before converting to .py file


In [ ]:
# ---------------------------------------------------------------------
# test inputs
# ---------------------------------------------------------------------

GeoDB = r"Y:\aa_GSS_Student_Engagement_Training\Sho\GeoDBToText\data\Peatland_Watershed_Protection_Areas_(WPAs)"

DefaultTitle = "Default Title"
DefaultTags = "roads, transportation"
DefaultSummary = "This is a summary."
DefaultDescription = "This is a description."
DefaultCredits = "Winona State University"
DefaultUseLimitations = "Public domain"
DefaultContact = "Sho Sato"
Delimiter = "|"

arcpy.env.workspace = GeoDB

fc = arcpy.ListFeatureClasses()[0]
ExportCSV(fc)